In [ ]:
import scipy.stats as stats
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import pandas as pd

In [ ]:
data = pd.read_parquet("/content/final_32_columns_no_outliers.parquet")

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

In [ ]:
data.columns

Index(['isRefundable', 'is_weekend', 'layover_proportion',
       'journey_start_hour', 'isBasicEconomy', 'seasonal_demand_impact',
       'is_holiday_season', 'travelDuration', 'start_airport_month_count',
       'dest_airport_month_count', 'elapsedDays', 'flight_month_cos',
       'layoverDurationMinutes', 'arrival_hour', 'arrival_minutes_of_day',
       'start_airport_daysbin_avg_fare', 'dest_airport_daysbin_avg_fare',
       'start_airport_month_avg_fare', 'dest_airport_month_avg_fare',
       'layover_discomfort_cost', 'flight_month_sin',
       'start_airport_daysbin_count', 'dest_airport_daysbin_count',
       'seatsRemaining', 'is_late_night_arrival', 'isNonStop',
       'totalTravelDistance', 'travel_efficiency', 'nonstop_premium_demand',
       'late_night_arrival_impact', 'numLayovers', 'numDaysToFlight',
       'totalFare'],
      dtype='object')

OLS

In [ ]:
# Define target
y = data['totalFare']

# Define features (drop target column)
X = data.drop(columns=['totalFare'])

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

Shape of X: (73233, 32)
Shape of y: (73233,)


In [ ]:
# Step 1: Baseline Multiple Linear Regression (MLR) using 32 predictors
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, linear_reset

# Remove the 3 low-contribution columns from the 35-feature set
cols_to_remove = ['is_mixed_itinerary', 'airline_', 'demand_pressure', 'layover_discomfort_cost','nonstop_premium_demand','travel_efficiency','seasonal_demand_impact','layover_proportion' ]
X_32 = X.drop(columns=cols_to_remove, errors='ignore').copy()

print(f"Initial features in X_final: {X.shape[1]}")
print(f"Features after dropping 3 columns: {X_32.shape[1]}")
print("Dropped columns:", cols_to_remove)

# Prepare clean regression frame
mlr_df = X_32.copy()
mlr_df['totalFare'] = y
mlr_df = mlr_df.replace([np.inf, -np.inf], np.nan).dropna()

X_mlr = mlr_df.drop(columns=['totalFare'])
y_mlr = mlr_df['totalFare']

# Fit OLS baseline0
X_mlr_const = sm.add_constant(X_mlr, has_constant='add')
ols_model = sm.OLS(y_mlr, X_mlr_const).fit()

# Save fitted values and residuals for diagnostics in next cells
y_hat = ols_model.fittedvalues
resid = ols_model.resid
std_resid = ols_model.get_influence().resid_studentized_internal

print(f"\nObservations used in OLS: {len(y_mlr):,}")
print(f"Predictors used in OLS: {X_mlr.shape[1]}")
print(f"Training R^2: {r2_score(y_mlr, y_hat):.4f}")
print(f"Training RMSE: {np.sqrt(mean_squared_error(y_mlr, y_hat)):.4f}")
print(f"Training MAE: {mean_absolute_error(y_mlr, y_hat):.4f}")

print("\nModel summary (top section):")
display(ols_model.summary().tables[0])

coef_df = ols_model.summary2().tables[1].sort_values('P>|t|')
print("Top 10 most statistically significant coefficients:")
display(coef_df.head(10))

Initial features in X_final: 32
Features after dropping 3 columns: 27
Dropped columns: ['is_mixed_itinerary', 'airline_', 'demand_pressure', 'layover_discomfort_cost', 'nonstop_premium_demand', 'travel_efficiency', 'seasonal_demand_impact', 'layover_proportion']

Observations used in OLS: 73,232
Predictors used in OLS: 27
Training R^2: 0.5144
Training RMSE: 131.9067
Training MAE: 95.8018

Model summary (top section):


Dep. Variable:,totalFare,R-squared:,0.514
Model:,OLS,Adj. R-squared:,0.514
Method:,Least Squares,F-statistic:,2872.
Date:,"Fri, 01 May 2026",Prob (F-statistic):,0.00
Time:,06:28:49,Log-Likelihood:,-4.6144e+05
No. Observations:,73232,AIC:,9.229e+05
Df Residuals:,73204,BIC:,9.232e+05
Df Model:,27,,
Covariance Type:,nonrobust,,


Top 10 most statistically significant coefficients:


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
isBasicEconomy,-148.570783,1.525961,-97.362111,0.000000e+00,-151.561661,-145.579905
numLayovers,91.696483,2.096424,43.739483,0.000000e+00,87.587500,95.805466
totalTravelDistance,0.074259,0.002304,32.231935,2.445386e-226,0.069743,0.078774
seatsRemaining,6.540437,0.210148,31.123070,2.831560e-211,6.128549,6.952326
is_weekend,34.343900,1.122021,30.608959,1.829122e-204,32.144742,36.543057
start_airport_month_avg_fare,0.570111,0.020108,28.352386,7.032827e-176,0.530700,0.609523
dest_airport_month_avg_fare,0.566789,0.021206,26.727755,1.271717e-156,0.525225,0.608353
const,-173.803748,7.066650,-24.594929,4.980287e-133,-187.654356,-159.953140
isNonStop,52.649415,2.229932,23.610324,8.711820e-123,48.278756,57.020073
isRefundable,2890.097364,131.958771,21.901518,5.512320e-106,2631.458650,3148.736078


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1) Residuals vs Fitted (non-linearity and funnel check)
axes[0].scatter(y_hat, resid, alpha=0.2, s=10)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)

# Quadratic trendline to detect U-shape pattern
x_grid = np.linspace(y_hat.min(), y_hat.max(), 300)
quad_coef = np.polyfit(y_hat, resid, deg=2)
quad_trend = np.polyval(quad_coef, x_grid)
axes[0].plot(x_grid, quad_trend, color='black', linewidth=2, label='Quadratic trend')
axes[0].set_title('Residuals vs Fitted')
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].legend()

# 2) Scale-Location plot (spread vs fitted)
axes[1].scatter(y_hat, np.sqrt(np.abs(std_resid)), alpha=0.2, s=10)
axes[1].set_title('Scale-Location Plot')
axes[1].set_xlabel('Fitted values')
axes[1].set_ylabel('Sqrt(|Standardized Residuals|)')

plt.tight_layout()
plt.savefig("residual_diagnostics.png", dpi=300)  # Save the figure for reference
plt.show()

# Formal diagnostics - Using ONLY Breusch-Pagan (Memory Safe)
bp_lm, bp_lm_pvalue, bp_f, bp_f_pvalue = het_breuschpagan(resid, X_mlr_const)

print("Non-linearity diagnostics")
print(f"Quadratic residual trend coefficient (x^2): {quad_coef[0]:.6f}")

print("\nHeteroscedasticity diagnostics")
print(f"Breusch-Pagan LM p-value: {bp_lm_pvalue:.6g}")
print(f"Breusch-Pagan F p-value : {bp_f_pvalue:.6g}")

# Additional practical signal for funnel-shape
corr_abs_resid_fit = np.corrcoef(np.abs(resid), y_hat)[0, 1]
print(f"Corr(|residual|, fitted): {corr_abs_resid_fit:.4f}")